In [0]:
# The cluster already knows the key from Spark config!
# You do NOT need any spark.conf.set() calls here.

path = "abfss://landing-zone@lshc.dfs.core.windows.net/clinic/"
display(dbutils.fs.ls(path))

In [0]:
# Storage variables
STORAGE_ACCOUNT = "lshc"
CONTAINER_NAME = "landing-zone"

# Cloud locations
base_adls = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
source_clinic_path = f"{base_adls}/clinic/"
bronze_delta_path = f"{base_adls}/delta/bronze_clinic_claims"

# Streaming metadata (checkpoints and schema inference)
checkpoint_csv = f"{base_adls}/checkpoints/bronze_clinic_csv"
schema_csv = f"{base_adls}/schema/bronze_clinic_csv"

checkpoint_parquet = f"{base_adls}/checkpoints/bronze_clinic_parquet"
schema_parquet = f"{base_adls}/schema/bronze_clinic_parquet"

print(f"Targeting source path: {source_clinic_path}")

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

# Read incoming CSVs using Auto Loader
df_csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_csv + "_v2")
    .option("pathGlobFilter", "*.csv")
    .option("header", "true")
    .load(source_clinic_path)
    .withColumn("_source_format", lit("CSV"))
    .withColumn("_ingested_file_name", input_file_name())
    .withColumn("_ingested_at", current_timestamp())
)

# Append to Bronze Delta Table
query_csv = (
    df_csv_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_csv)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start(bronze_delta_path)
)

query_csv.awaitTermination()
print("CSV batch ingested successfully into Bronze Delta table.")

In [0]:
# Read incoming Parquet files using Auto Loader
df_parquet_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_parquet)
    .option("pathGlobFilter", "*.parquet")
    .load(source_clinic_path)
    .withColumn("_source_format", lit("PARQUET"))
    .withColumn("_ingested_file_name", input_file_name())
    .withColumn("_ingested_at", current_timestamp())
)

# Append to the same Bronze Delta Table
query_parquet = (
    df_parquet_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_parquet)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start(bronze_delta_path)
)

query_parquet.awaitTermination()
print("Parquet batch ingested successfully into Bronze Delta table.")

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS healthcare_lakehouse;
USE healthcare_lakehouse;

CREATE TABLE IF NOT EXISTS bronze_clinic_claims
USING DELTA
LOCATION 'abfss://landing-zone@lshc.dfs.core.windows.net/delta/bronze_clinic_claims';

-- Check total record counts by source format
SELECT 
    _source_format, 
    COUNT(*) AS total_records 
FROM bronze_clinic_claims 
GROUP BY _source_format;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_lakehouse.bronze_clinic_claims
USING DELTA
LOCATION 'abfss://landing-zone@lshc.dfs.core.windows.net/delta/bronze_clinic_claims';

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%sql
USE CATALOG default;
USE SCHEMA default;

In [0]:
# Read the Bronze Delta files directly from your ADLS storage
bronze_path = "abfss://landing-zone@lshc.dfs.core.windows.net/delta/bronze_clinic_claims"
df_bronze = spark.read.format("delta").load(bronze_path)

# Verify count
print(f"Total rows in Bronze: {df_bronze.count()}")

# Inspect the records
display(df_bronze.limit(10))

In [0]:
# Create a session-level view pointing to the Delta DataFrame
df_bronze.createOrReplaceTempView("bronze_clinic_claims")
print("Temporary view 'bronze_clinic_claims' is ready for SQL querying.")

In [0]:
%sql
SELECT 
    claim_id,
    patient_id,
    claim_amount,
    diagnosis_code,
    _source_format,
    _ingested_at
FROM bronze_clinic_claims


In [0]:
%sql
SELECT 
    _source_format,
    COUNT(*) AS total_records,
    ROUND(AVG(claim_amount), 2) AS avg_claim_amount,
    COUNT(CASE WHEN try_cast(claim_amount AS DOUBLE) < 0 THEN 1 END) AS negative_claims_count
FROM bronze_clinic_claims
GROUP BY _source_format;

In [0]:
%sql
select distinct claim_id
from bronze_clinic_claims